# Binary Label Transfer Learning


In [14]:
# !pip install datasets transformers scikit-learn # For Colab
%pip install torch
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports


In [15]:
# This quick check tells us whether PyTorch can see a GPU.
# Training DistilBERT on CPU can still work, but it will be much slower.
# If this prints True and shows a GPU name, the training cell should finish much faster.
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


GPU available: False


In [16]:
# Basic libraries for tables, numbers, and model training.
# pandas is used for CSV files, numpy is used for metric calculations,
# torch is used by the model, and inspect lets us handle small version
# differences in Hugging Face TrainingArguments.
import pandas as pd
import numpy as np
import torch
import inspect

# Hugging Face tools.
# Dataset converts pandas tables into the format expected by Trainer.
# AutoTokenizer turns comments into token IDs.
# AutoModelForSequenceClassification loads DistilBERT with a classification head.
# TrainingArguments and Trainer handle the training loop for us.
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Metrics for binary classification.
# These are the same main metrics used in the multi-label notebook, with accuracy
# and a confusion matrix added because the task is now one clean/unclean decision.
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


OSError: Could not load this library: C:\Users\ajayi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchaudio\lib\libtorchaudio.pyd

## Load Data And Create Binary Label


In [ ]:
# Load the same three CSV files used in the multi-label transfer-learning notebook.
# train.csv has comments and the six original toxicity labels.
# test.csv has comments for the test set.
# test_labels.csv has the real labels for the test comments, but some rows contain -1
# because Kaggle did not score those rows. We remove those -1 rows later before testing.
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Test labels shape:", test_labels.shape)

# These are the original six labels from the dataset.
# The binary notebook still starts from these columns, but it combines them into one target.
# If any of these labels is 1, the comment is treated as unclean.
# If all six labels are 0, the comment is treated as clean.
target_cols = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Count how many toxicity labels each training comment has.
# A clean comment has label_count = 0.
# An unclean comment has label_count >= 1.
train_df["label_count"] = train_df[target_cols].sum(axis=1)
train_df["unclean"] = (train_df["label_count"] > 0).astype(int)
train_df["clean"] = 1 - train_df["unclean"]

print("Clean comments:", train_df["clean"].sum())
print("Unclean comments:", train_df["unclean"].sum())
print("Unclean percent:", round(train_df["unclean"].mean() * 100, 2))

train_df.head()


## Split Training And Validation Data


In [ ]:
# Split the original training data into training and validation parts.
# The original multi-label notebook used a plain random split.
# Here we can stratify because the target is one binary column.
# Stratify keeps the clean/unclean ratio almost the same in both splits,
# which is helpful because only about 10% of the comments are unclean.
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["unclean"]
)

# Reset the index so the rows are numbered from 0 again after splitting.
# This is not required for the model, but it keeps the tables cleaner when we inspect them.
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Training data:", train_data.shape)
print("Validation data:", val_data.shape)
print("Training unclean percent:", round(train_data["unclean"].mean() * 100, 2))
print("Validation unclean percent:", round(val_data["unclean"].mean() * 100, 2))


## Convert Pandas DataFrames To Hugging Face Datasets


In [ ]:
# Hugging Face Trainer works best with a Dataset object instead of a pandas DataFrame.
# We keep all columns for now because the preprocessing function still needs comment_text
# and the new binary label column called unclean.
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

print(train_dataset.column_names)


## Load DistilBERT Tokenizer


In [ ]:
# Use the same pretrained model as the multi-label transfer-learning notebook.
# This keeps the comparison fair: the main change is the target, not the language model.
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


## Tokenize Text And Prepare Binary Labels


In [ ]:
def preprocess_data(batch):
    """
    Tokenize comments and create the binary label used by the model.

    The tokenizer changes each comment into numbers that DistilBERT understands.
    The label is stored as a one-item list, for example [0.0] or [1.0].
    That shape matters because the model will output one logit for each comment.
    One logit means one raw score for the question: is this comment unclean?
    """

    # Turn comment text into token IDs and attention masks.
    # truncation=True cuts very long comments so they fit the model input size.
    # padding="max_length" makes every row the same length, which allows batching.
    # max_length=128 is kept from the multi-label notebook so the binary version changes
    # as little as possible from the original transfer-learning setup.
    encoding = tokenizer(
        batch["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # Build labels for binary classification.
    # The model expects floats because we are using BCEWithLogitsLoss internally.
    # 0.0 means clean and 1.0 means unclean.
    labels = []
    for value in batch["unclean"]:
        labels.append([float(value)])

    # Trainer looks for a column named labels.
    # When Trainer passes a batch to the model, these labels are compared with the logits
    # to calculate the training loss.
    encoding["labels"] = labels

    return encoding


## Apply Preprocessing


In [ ]:
# Apply the same preprocessing function to training and validation data.
# remove_columns deletes the original text and label columns after they are converted.
# This leaves only the tensors the model needs, such as input_ids, attention_mask, and labels.
train_dataset = train_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=val_dataset.column_names
)


In [ ]:
# After preprocessing, the dataset should contain token columns and labels.
# The exact token columns can vary a little by tokenizer, so we print them to confirm.
print(train_dataset.column_names)
print(val_dataset.column_names)


## Set Dataset Format For PyTorch


In [ ]:
# Convert the dataset columns into PyTorch tensors.
# This is the format the DistilBERT model expects during training and evaluation.
train_dataset.set_format("torch")
val_dataset.set_format("torch")


## Load Pretrained DistilBERT Model


In [ ]:
# There is only one output now: unclean.
# The model gives one raw number, called a logit, for each comment.
# A larger logit means the model is more confident the comment is unclean.
id2label = {
    0: "unclean"
}

label2id = {
    "unclean": 0
}

# num_labels=1 makes the classification head return one score instead of six.
# problem_type="multi_label_classification" tells Hugging Face to use BCEWithLogitsLoss.
# This is useful for a one-output binary setup because the loss applies sigmoid internally
# and compares the result with labels shaped like [0.0] or [1.0].
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)


## Define Evaluation Metrics


In [ ]:
def compute_metrics(eval_pred):
    """
    Calculate binary classification metrics at the default threshold of 0.5.

    Trainer calls this function during validation and testing.
    The model returns logits, so we first apply sigmoid to turn them into probabilities.
    Then we use threshold 0.5 to make clean/unclean predictions.
    Later in the notebook we search for a better threshold using validation F1.
    """

    logits, labels = eval_pred

    # Flatten both arrays because this binary model has one output per comment.
    # After flattening, probabilities, labels, and predictions are simple one-dimensional arrays.
    logits = logits.reshape(-1)
    labels = labels.reshape(-1).astype(int)

    # Sigmoid converts any raw logit into a probability between 0 and 1.
    # Example: 0.92 means the model thinks the comment is very likely unclean.
    probabilities = 1 / (1 + np.exp(-logits))

    # Use the normal 0.5 threshold during Trainer evaluation.
    # If probability is 0.5 or higher, predict unclean.
    # Otherwise, predict clean.
    predictions = (probabilities >= 0.5).astype(int)

    # ROC-AUC can fail if a split somehow contains only one class.
    # That should not happen here because we stratified the split, but the try/except
    # makes the function safer and easier to reuse.
    try:
        roc_auc = roc_auc_score(labels, probabilities)
    except ValueError:
        roc_auc = np.nan

    # The confusion matrix shows the actual mistakes, not just one score.
    # labels=[0, 1] fixes the row/column order as clean first, unclean second.
    cm = confusion_matrix(labels, predictions, labels=[0, 1])

    return {
        "roc_auc": roc_auc,
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "clean_correct": cm[0, 0],
        "clean_called_unclean": cm[0, 1],
        "unclean_called_clean": cm[1, 0],
        "unclean_correct": cm[1, 1]
    }


## Define Training Settings


In [ ]:
# These settings are almost the same as the multi-label transfer-learning notebook.
# The output folder is changed so the binary model does not overwrite the multi-label model.
# The best model is selected by ROC-AUC because ROC-AUC checks how well the model ranks
# unclean comments above clean comments before we choose a final threshold.
args_dict = {
    "output_dir": "./distilbert-toxic-binary-model",
    "save_strategy": "epoch",
    "num_train_epochs": 2,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "roc_auc",
    "greater_is_better": True,
    "report_to": "none",
    "fp16": torch.cuda.is_available()
}

# Hugging Face changed the argument name in different versions.
# Some versions expect eval_strategy and older versions expect evaluation_strategy.
# This check keeps the notebook from breaking just because the installed Transformers
# version uses one name instead of the other.
training_args_params = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in training_args_params:
    args_dict["eval_strategy"] = "epoch"
else:
    args_dict["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_dict)


## Create Trainer


In [ ]:
# Trainer connects the model, training settings, datasets, and metric function.
# The actual training still happens in the next cell, but this cell prepares everything.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


In [ ]:
# Start training.
# Trainer will evaluate after each epoch and keep the checkpoint with the best validation ROC-AUC.
trainer.train()


## Evaluate On Validation Data


In [ ]:
# Evaluate the validation set with the default 0.5 threshold.
# This is useful as a first check, but because the dataset is imbalanced we also search
# for a better threshold in a later cell.
val_results = trainer.evaluate()

print(val_results)


In [ ]:
# Merge the test comments with their labels.
# test.csv has the text and test_labels.csv has the labels, so we join them by id.
# Rows with -1 labels are not real scored examples, so they are removed before testing.
test_full = test_df.merge(test_labels, on="id")
test_full = test_full[(test_full[target_cols] != -1).all(axis=1)].copy()

# Create the same binary label for test data that we created for training data.
# This keeps the target definition exactly the same: any original toxic label means unclean.
test_full["label_count"] = test_full[target_cols].sum(axis=1)
test_full["unclean"] = (test_full["label_count"] > 0).astype(int)
test_full["clean"] = 1 - test_full["unclean"]

test_full = test_full.reset_index(drop=True)

print("Usable test data:", test_full.shape)
print("Clean test comments:", test_full["clean"].sum())
print("Unclean test comments:", test_full["unclean"].sum())


In [ ]:
# Convert the prepared test table into a Hugging Face Dataset.
# Then apply the same tokenizer and label formatting used for training and validation.
test_dataset = Dataset.from_pandas(test_full)

test_dataset = test_dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=test_dataset.column_names
)

test_dataset.set_format("torch")

print(test_dataset.column_names)


## Evaluate On Test Data With Default Threshold


In [ ]:
# Evaluate the test dataset with the default 0.5 threshold.
# This gives a direct comparison with the validation result above.
# The next section finds a threshold using validation data and then applies that threshold to test.
test_results = trainer.evaluate(
    eval_dataset=test_dataset
)

print(test_results)


## Find Best Binary Threshold On Validation Data


In [ ]:
# Get raw model predictions on the validation data.
# We use validation predictions to choose the threshold because the test set should be used
# only for the final score. This avoids tuning the threshold directly on test results.
val_output = trainer.predict(val_dataset)

val_logits = val_output.predictions.reshape(-1)
val_labels = val_output.label_ids.reshape(-1).astype(int)

# Convert logits to probabilities.
val_probs = 1 / (1 + np.exp(-val_logits))


In [ ]:
# Search for the threshold that gives the best validation F1 score.
# Lower thresholds usually catch more unclean comments but can create more false positives.
# Higher thresholds usually reduce false positives but can miss more unclean comments.
# F1 is a good balance here because it considers both precision and recall.
best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.05, 0.96, 0.01):
    preds = (val_probs >= threshold).astype(int)
    f1 = f1_score(val_labels, preds, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("Best threshold:", round(best_threshold, 2))
print("Best validation F1:", round(best_f1, 4))


In [ ]:
# Get raw model predictions on the test data.
# We do not choose the threshold here. We only apply the threshold already chosen
# from the validation set above.
test_output = trainer.predict(test_dataset)

test_logits = test_output.predictions.reshape(-1)
test_labels_np = test_output.label_ids.reshape(-1).astype(int)

test_probs = 1 / (1 + np.exp(-test_logits))
test_preds = (test_probs >= best_threshold).astype(int)


In [ ]:
# Calculate final binary test metrics using the validation-selected threshold.
# ROC-AUC uses probabilities because it checks ranking quality.
# Accuracy, precision, recall, and F1 use final clean/unclean predictions.
test_roc_auc = roc_auc_score(test_labels_np, test_probs)
test_accuracy = accuracy_score(test_labels_np, test_preds)
test_precision = precision_score(test_labels_np, test_preds, zero_division=0)
test_recall = recall_score(test_labels_np, test_preds, zero_division=0)
test_f1 = f1_score(test_labels_np, test_preds, zero_division=0)

print("Test ROC-AUC:", test_roc_auc)
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test F1:", test_f1)


In [ ]:
# Build a confusion matrix so the mistakes are easy to understand.
# Rows show the real label. Columns show what the model predicted.
# The most important number to watch is Actual unclean / Predicted clean,
# because those are toxic comments the model missed.
cm = confusion_matrix(test_labels_np, test_preds, labels=[0, 1])

confusion_df = pd.DataFrame(
    cm,
    index=["Actual clean", "Actual unclean"],
    columns=["Predicted clean", "Predicted unclean"]
)

confusion_df


In [ ]:
# Store the final results in one small table for the report.
# This table is easier to copy into the write-up than separate print statements.
final_results = pd.DataFrame([
    {
        "threshold": best_threshold,
        "roc_auc": test_roc_auc,
        "accuracy": test_accuracy,
        "precision": test_precision,
        "recall": test_recall,
        "f1": test_f1,
        "clean_correct": cm[0, 0],
        "clean_called_unclean": cm[0, 1],
        "unclean_called_clean": cm[1, 0],
        "unclean_correct": cm[1, 1]
    }
])

final_results
